Imports

In [22]:

import numpy as np

from Background_functions import read_audio_file, split_audio_segments, start_end_times
from STFT import resample_audio, stft_calculation, plot_spectrogram
from DFT import dp

from pathlib import Path
from scipy.spatial import distance as dist

DATA_22 = Path(r"C:\Users\HP\Desktop\Skripsie data\DataSubmission\2023_04_22")

Data download

In [23]:
audio = DATA_22 / "20230422_171301.WAV"
f_s, x = read_audio_file(audio)
duration = len(x) / f_s  # Duration of the audio in seconds

start = 0.0  # Start time in seconds
window = 3.0  # Window length in seconds
start_t = []
end_t = []

while start < duration:
    end = min(start + window, duration)  # Ensure we don't exceed the audio duration
    start_t.append(start)
    end_t.append(end)
    start += window  # Move to the next window
length = len(start_t)
print(length)

audio_segments = split_audio_segments(x, f_s, start_t, end_t)

89479


Template

In [34]:
text = DATA_22 / "20230422_171301.Detections.selections.txt"
temp_marks, start_t_temp, end_t_temp = start_end_times(text)

whale_call_segments = split_audio_segments(x, f_s, start_t_temp, end_t_temp)
print(len(whale_call_segments))
audio_template_1 = whale_call_segments[0]

1262


DTW Model

In [25]:
def DTW_calc(template_Zxx, comparison_Zxx):
    x_seq = np.abs(template_Zxx).T      # shape: (n_time_frames, n_freq_bins)
    y_seq = np.abs(comparison_Zxx).T    # shape: (n_time_frames, n_freq_bins)

    dist_mat = dist.cdist(x_seq, y_seq, "cosine")
    path, cost_mat = dp(dist_mat)
    ali_cost = cost_mat[-1, -1]
    #print("Alignment cost: {:.4f}".format(ali_cost))

    M = x_seq.shape[0]
    N = y_seq.shape[0]
    norm_ali_cost = ali_cost / (M + N)
    #print("Normalized alignment cost: {:.4f}".format(norm_ali_cost))
    return norm_ali_cost

STFT Model

In [26]:
def short_time_calc(temp, sig, fs):
    fs_new = 2000
    resampled_sig = resample_audio(sig, fs, fs_new)
    resampled_temp = resample_audio(temp, fs, fs_new)
    f_seg, t_seg, Zxx_seg = stft_calculation(resampled_sig, fs, fs_new)
    f_temp, t_temp, Zxx_temp = stft_calculation(resampled_temp, fs, fs_new)
    #plot_spectrogram(f_seg, t_seg, Zxx_seg, fs_new)
    return Zxx_seg, Zxx_temp

Running loop for STFT template analysis

In [35]:
stft_cost = []
for i in range(1262):
    audio_seg = whale_call_segments[i]
    Zxx_seg, Zxx_temp = short_time_calc(audio_template_1, audio_seg, f_s)
    stft_cost.append(DTW_calc(Zxx_temp, Zxx_seg))
    #print(i)
avg_cost = np.mean(stft_cost)
max_cost = np.max(stft_cost)
min_cost = np.min(stft_cost)
print(f"avg_cost: {avg_cost:.2f}")
print(f"max_cost: {max_cost:.2f}")
print(f"min_cost: {min_cost:.2f}")

avg_cost: 0.12
max_cost: 0.55
min_cost: 0.00


Running loop for comparison

In [28]:
stft_cost = []
for i in range(100):
    audio_seg = audio_segments[i]
    Zxx_seg, Zxx_temp = short_time_calc(audio_template_1, audio_seg, f_s)
    stft_cost.append(DTW_calc(Zxx_temp, Zxx_seg))
    #print(i)
